# Week 3 • Day 2: Hugging Face Pipelines for Quick AI Inference (Lec 68–71)

Welcome to **Week 3 Day 2**! Today we explore Hugging Face's high-level **Pipelines API** for running fast, turnkey open-source AI inference.

### 🌟 Lectures Covered:
1. **Lec 68**: Introduction to Hugging Face Pipelines for Quick AI Inference (Training vs. Inference)
2. **Lec 69**: Text NLP Pipelines: Sentiment Analysis, Named Entity Recognition & Question Answering
3. **Lec 70**: Sequence-to-Sequence NLP: Summarization, Translation, Zero-Shot Classification & Text Generation
4. **Lec 71**: Multi-Modal Pipelines: Image Generation (SDXL-Turbo) & Audio Synthesis (SpeechT5)

---

## 🎓 Lec 68: Environment Setup & Pipeline Foundations
Install the required libraries and verify hardware acceleration.

In [ ]:
# Core open-source AI libraries for Pipelines
!pip install -q --upgrade datasets==3.6.0 transformers==4.57.6 huggingface_hub diffusers soundfile accelerate

In [ ]:
# Check T4 GPU connection
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0 or 'command not found' in gpu_info:
    print('⚠️ Running on CPU mode. (To use GPU: Runtime > Change runtime type > T4 GPU)')
else:
    print('✅ GPU detected! Full output below:')
    print(gpu_info)

In [ ]:
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline
from diffusers import AutoPipelineForText2Image
from datasets import load_dataset
from IPython.display import Audio, display

# Authenticate with Hugging Face Hub
try:
    hf_token = userdata.get('HF_TOKEN')
    login(hf_token, add_to_git_credential=True)
    print('✅ Successfully authenticated with Hugging Face Hub!')
except Exception as e:
    print(f'Please ensure HF_TOKEN is in Colab Secrets: {e}')

## 📝 Lec 69: Text NLP Pipelines (Sentiment, NER & QA)
Exploring common natural language tasks with default and specialized models.

In [ ]:
# 1. Binary Sentiment Analysis (Default DistilBERT)
sentiment_analyzer = pipeline("sentiment-analysis", device="cuda" if torch.cuda.is_available() else "cpu")
print(sentiment_analyzer("I'm super excited to be on the way to LLM mastery!"))
print(sentiment_analyzer("I should be more excited to be on the way to LLM mastery!"))

In [ ]:
# 2. Multilingual 5-Star Sentiment (BERT)
better_sentiment = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device="cuda" if torch.cuda.is_available() else "cpu"
)
result = better_sentiment("I should be more excited to be on the way to LLM mastery!!")
print(result)

In [ ]:
# 3. Named Entity Recognition (NER)
ner = pipeline("ner", device="cuda" if torch.cuda.is_available() else "cpu")
entities = ner("AI Engineers are learning about the amazing pipelines from HuggingFace in Google Colab from Ed Donner")
for entity in entities:
    print(f"{entity['word']:<15} | Type: {entity['entity']:<8} | Confidence: {entity['score']:.4f}")

In [ ]:
# 4. Extractive Question Answering with Context
question_answerer = pipeline("question-answering", device="cuda" if torch.cuda.is_available() else "cpu")
qa_result = question_answerer(
    question="What are Hugging Face pipelines?",
    context="Pipelines are a high level API for inference of LLMs with common tasks"
)
print(qa_result)

## 🔄 Lec 70: Sequence-to-Sequence NLP (Summarization, Translation & Zero-Shot)
More complex multi-token tasks converting sequences of text to sequences of output.

In [ ]:
# 1. Text Summarization
summarizer = pipeline("summarization", device="cuda" if torch.cuda.is_available() else "cpu")
article = """
The Hugging Face transformers library is an incredibly versatile and powerful tool for natural language processing (NLP).
It allows users to perform a wide range of tasks such as text classification, named entity recognition, and question answering, among others.
It's an extremely popular library that's widely used by the open-source data science community.
It lowers the barrier to entry into the field by providing Data Scientists with a productive, convenient way to work with transformer models.
"""
summary = summarizer(article, max_length=50, min_length=25, do_sample=False)
print(summary[0]['summary_text'])

In [ ]:
# 2. Translation (English to French & Spanish)
device_choice = "cuda" if torch.cuda.is_available() else "cpu"
text_to_translate = "The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API."

trans_fr = pipeline("translation_en_to_fr", device=device_choice)
print("French:", trans_fr(text_to_translate)[0]['translation_text'])

trans_es = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es", device=device_choice)
print("Spanish:", trans_es(text_to_translate)[0]['translation_text'])

In [ ]:
# 3. Zero-Shot Classification (NLI)
classifier = pipeline("zero-shot-classification", device="cuda" if torch.cuda.is_available() else "cpu")
zero_shot_res = classifier(
    "Hugging Face's Transformers library is amazing!",
    candidate_labels=["technology", "sports", "politics"]
)
print(zero_shot_res)

In [ ]:
# 4. Causal Text Generation
generator = pipeline("text-generation", device="cuda" if torch.cuda.is_available() else "cpu")
gen_res = generator("If there's one thing I want you to remember about using HuggingFace pipelines, it's", max_length=50)
print(gen_res[0]['generated_text'])

## 🎨 Lec 71: Multi-Modal Pipelines (Image & Audio)
Extending pipelines to diffusion image models and text-to-speech.

In [ ]:
# 1. Diffusion Image Generation (SDXL-Turbo in 4 steps)
if torch.cuda.is_available():
    pipe_img = AutoPipelineForText2Image.from_pretrained(
        "stabilityai/sdxl-turbo",
        torch_dtype=torch.float16,
        variant="fp16"
    ).to("cuda")
    prompt = "A class of students learning AI engineering in a vibrant pop-art style"
    image = pipe_img(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
    display(image)
else:
    print("SDXL-Turbo requires CUDA GPU to execute.")

In [ ]:
# 2. Audio Synthesis (SpeechT5 with Speaker Embeddings)
synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device="cuda" if torch.cuda.is_available() else "cpu")
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)

speech = synthesiser(
    "Hi to an artificial intelligence engineer, on the way to mastery!",
    forward_params={"speaker_embeddings": speaker_embedding}
)
Audio(speech["audio"], rate=speech["sampling_rate"])